### Бізнес-контекст: Аудит залишків та продажів нових ноутбуків

Уявіть, що ви керуєте категорією ноутбуків. Постачальник привіз нову лінійку пристроїв. Вам потрібно швидко проаналізувати перші результати продажів за тиждень та виявити проблеми з поставками.

У вас є три джерела сирих даних (наприклад, вивантажені з ERP-системи у вигляді списків):

Довідник новинок (dict) — список моделей, які приїхали в новій партії, та їхня рекомендована роздрібна ціна (РРЦ).

Продажі за тиждень (list із tuple) — лог транзакцій (що саме і в якій кількості купили).

Фактична наявність на складах (set) — артикули, які фізично залишилися на складі на вечір неділі.

In [1]:
# 1. Словник: Артикул -> РРЦ (грн)
new_laptops_catalog = {
    'NB-ACER-N5-01': 42000, 
    'NB-ACER-A7-02': 35000, 
    'NB-ACER-S3-03': 29000, 
    'NB-ACER-S5-04': 48000, 
    'NB-ACER-G9-05': 85000
}

In [2]:
new_laptops_catalog

{'NB-ACER-N5-01': 42000,
 'NB-ACER-A7-02': 35000,
 'NB-ACER-S3-03': 29000,
 'NB-ACER-S5-04': 48000,
 'NB-ACER-G9-05': 85000}

In [3]:
# 2. Список кортежів: (Артикул, Кількість у чеку(
weekly_sales_log = [
    ("NB-ACER-N5-01", 2),
    ("NB-ACER-A7-02", 1),
    ("NB-ACER-N5-01", 1),
    ("NB-ACER-S3-03", 5),
    ("NB-ACER-N5-01", 1),
    ("NB-ACER-A7-02", 2),
    # Опа, а тут менеджер помилився і вбив старий артикул, якого немає в каталозі новинок:
    ("NB-ASUS-X5-99", 1) 
]

In [4]:
# 3. Множина: Артикули, які фактично є на складі в кінці тижня
current_warehouse_stock = {"NB-ACER-N5-01", "NB-ACER-A7-02", "NB-ACER-G9-05"}

In [5]:
# Задача 1: Очищення логів та підрахунок виторгу
total_sum = 0
for pn, price in weekly_sales_log:
   if pn in new_laptops_catalog:
       subtotal = new_laptops_catalog[pn] * price
       total_sum += subtotal
print(f'Загальна сума виторгу: {total_sum}')

Загальна сума виторгу: 418000


In [6]:
# Задача 2: Агрегація продажів (Аналог groupby)

# Варіант 1: Класичний (через if-else)
agregated_sales1 = {}

for pn, qty in weekly_sales_log:
    # Відфільтруємо відразу старі товари
    if pn in new_laptops_catalog:
        if pn not in agregated_sales1:
            # Якщо товару ще нема в нашому звіті - створюємо ключ
            agregated_sales1[pn] = qty
        else: 
            # Якщо товар уже є - плюсуємо кількість до старої
            agregated_sales1[pn] += qty
print(agregated_sales1)

    

{'NB-ACER-N5-01': 4, 'NB-ACER-A7-02': 3, 'NB-ACER-S3-03': 5}


In [8]:
# Варіант 2: Професійний (через метод .get())
aggregated_sales2 = {}

for pn, qty in weekly_sales_log:
    if pn in new_laptops_catalog:
        # Якщо pn немає в словнику, .get() поверне 0, і ми додамо qty.
        # Якщо pn є, .get() поверне поточну суму, і ми додамо qty.
        aggregated_sales2[pn] = aggregated_sales2.get(pn, 0) + qty
print(aggregated_sales2)
        

{'NB-ACER-N5-01': 4, 'NB-ACER-A7-02': 3, 'NB-ACER-S3-03': 5}


In [9]:
# Задача 3: Аналіз "зависшого" товару (Множини)
catalog_set = set(new_laptops_catalog)

In [10]:
catalog_set

{'NB-ACER-A7-02',
 'NB-ACER-G9-05',
 'NB-ACER-N5-01',
 'NB-ACER-S3-03',
 'NB-ACER-S5-04'}

In [12]:
aggregated_sales_set = set(aggregated_sales2)

In [13]:
aggregated_sales_set

{'NB-ACER-A7-02', 'NB-ACER-N5-01', 'NB-ACER-S3-03'}

In [14]:
catalog_set.intersection(current_warehouse_stock)

{'NB-ACER-A7-02', 'NB-ACER-G9-05', 'NB-ACER-N5-01'}

In [15]:
catalog_set.intersection(current_warehouse_stock) - aggregated_sales_set

{'NB-ACER-G9-05'}

In [16]:
dificit_set = agregated_sales_set - current_warehouse_stock

In [17]:
dificit_set

{'NB-ACER-S3-03'}